In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings("ignore")
print("✅ Imports done!")

✅ Imports done!


In [2]:
df = pd.read_csv("C:/Users/Gazala Anjum B K/Documents/cyber threat detection/cyber-threat-detection/data/features.csv")
with open("C:/Users/Gazala Anjum B K/Documents/cyber threat detection/cyber-threat-detection/data/class_weights.json") as f:
    class_weights = json.load(f)
X = df.drop("label", axis=1)
y = df["label"]
print(f"Features shape: {X.shape}")
print(f"Class weights: {class_weights}")

Features shape: (641119, 33)
Class weights: {'0': 0.7488308260138292, '1': 1.5046986701965368}


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")

Training set: (512895, 33)
Test set:     (128224, 33)


In [4]:
print("Training Random Forest...")
print("This takes 3-5 minutes...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1, class_weight={0: class_weights["0"], 1: class_weights["1"]})
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print(f"\n✅ Random Forest trained! Accuracy: {rf_acc*100:.2f}%")
print(classification_report(y_test, rf_pred, target_names=["Safe", "Malicious"]))

Training Random Forest...
This takes 3-5 minutes...

✅ Random Forest trained! Accuracy: 95.79%
              precision    recall  f1-score   support

        Safe       0.98      0.96      0.97     85616
   Malicious       0.92      0.95      0.94     42608

    accuracy                           0.96    128224
   macro avg       0.95      0.96      0.95    128224
weighted avg       0.96      0.96      0.96    128224



In [5]:
print("Training XGBoost...")
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, scale_pos_weight=scale_pos_weight, eval_metric="logloss", verbosity=0)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)
print(f"\n✅ XGBoost trained! Accuracy: {xgb_acc*100:.2f}%")
print(classification_report(y_test, xgb_pred, target_names=["Safe", "Malicious"]))

Training XGBoost...

✅ XGBoost trained! Accuracy: 94.46%
              precision    recall  f1-score   support

        Safe       0.97      0.95      0.96     85616
   Malicious       0.90      0.94      0.92     42608

    accuracy                           0.94    128224
   macro avg       0.93      0.94      0.94    128224
weighted avg       0.95      0.94      0.94    128224



In [6]:
os.makedirs("C:/Users/Gazala Anjum B K/Documents/cyber threat detection/cyber-threat-detection/models", exist_ok=True)
joblib.dump(rf_model, "C:/Users/Gazala Anjum B K/Documents/cyber threat detection/cyber-threat-detection/models/random_forest.pkl")
joblib.dump(xgb_model, "C:/Users/Gazala Anjum B K/Documents/cyber threat detection/cyber-threat-detection/models/xgboost.pkl")
print("✅ Both models saved!")
print(f"\nRandom Forest: {rf_acc*100:.2f}%")
print(f"XGBoost:       {xgb_acc*100:.2f}%")

✅ Both models saved!

Random Forest: 95.79%
XGBoost:       94.46%


In [7]:
RF_WEIGHT  = 0.55
XGB_WEIGHT = 0.45
rf_proba       = rf_model.predict_proba(X_test)[:, 1]
xgb_proba      = xgb_model.predict_proba(X_test)[:, 1]
ensemble_proba = RF_WEIGHT * rf_proba + XGB_WEIGHT * xgb_proba
ensemble_pred  = (ensemble_proba >= 0.5).astype(int)
ensemble_acc = accuracy_score(y_test, ensemble_pred)
print(f"Random Forest: {rf_acc*100:.2f}%")
print(f"XGBoost:       {xgb_acc*100:.2f}%")
print(f"Ensemble:      {ensemble_acc*100:.2f}%")
ensemble_config = {"rf_weight": RF_WEIGHT, "xgb_weight": XGB_WEIGHT, "rf_accuracy": rf_acc, "xgb_accuracy": xgb_acc, "ensemble_accuracy": ensemble_acc, "threshold": 0.5}
with open("C:/Users/Gazala Anjum B K/Documents/cyber threat detection/cyber-threat-detection/models/ensemble_config.json","w") as f:
    json.dump(ensemble_config, f, indent=2)
print("\n✅ STEP 4 + 5 COMPLETE!")

Random Forest: 95.79%
XGBoost:       94.46%
Ensemble:      95.46%

✅ STEP 4 + 5 COMPLETE!
